# TP N°3 — k‑NN  & Indexation vectorielle (FAISS) 


Ce TP se concentre uniquement sur **la recherche de voisins** et **l’indexation** dans FAISS.
Les modèles d’embeddings ont été vus dans les TP1–TP2 : ici, on **part d’un tableau d’embeddings déjà calculés**.  
## 🎯 Objectifs pédagogiques
- **Partie A** : comparer la recherche *force brute* (cosinus/produit scalaire) à un **Index Flat (IndexFlatIP/L2)** et mesurer le **temps de requête**.
- **Partie B** : construire un index **IVF (Inverted File Index)**, régler `nlist`/`nprobe`, et **évaluer le compromis** *vitesse ↔ recall@k*.

À la fin du TP, vous saurez :
- Quand l’Index Flat devient avantageux face au calcul naïf *brute-force*.
- Comment **paramétrer** et **évaluer** un index IVF (rappel@k, latence).


> 💡 **Travail à faire** : Exécutez le notebook cellule par cellule et répondez aux questions (compléter le code, explications). Les corrections seront collectives avec partage d'écran pendant les séances. Lisez bien les commentaires et les questions.


### ⚙️ Préambule — Environnement (threads / tokenizers) 

In [ ]:
# Limitation du  nombre de threads (OMP/MKL) 
# et désactivation du parallélisme des tokenizers pour éviter 
# que le notebook monopolise la machine 
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


: 

In [ ]:
# Imports communs
import time, numpy as np, pandas as pd
from pprint import pprint
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import random, itertools
from collections import OrderedDict

import faiss
np.random.seed(42)


## 🧩 Format du TP
  
Certaines cellules de code sont juste à tester, parfois en faisant varier certains paramètres (mention # A COMPLETER), d'autres sont à compléter en ajoutant des commentaires (mention # A COMPLETER par des commentaires).


---
# Partie A — FAISS Index Flat 


### 1- Comparaison IndexFlatL2 et IndexFlatIP (avec et sans normalisation)  
Objectif: comprendre comment fonctionnent ces index sur un exemple simple (corpus synthétique)  

Exécutez le code suivant et répondez aux questions :  
**1️⃣.** Comparez les Top-3 obtenus par les trois index.  
→ Les indices (`id`) sont-ils les mêmes ?

**2️⃣.** Pourquoi les valeurs sont-elles **croissantes** pour `IndexFlatL2` mais **décroissantes** pour `IndexFlatIP` ?  

**3️⃣.** Quelle est la différence de sens entre une **distance euclidienne** et un **produit scalaire** ?  

**4️⃣.** Pourquoi la **normalisation L2** transforme-t-elle le produit scalaire en **similariré cosinus de l’angle** entre vecteurs ?

**5️⃣.** Quelles sont les plages de valeurs observées pour : les distances L2, les scores IndexFlatIP (avec et sans normalisation)?  

**6️⃣.** Si deux vecteurs ont la même direction mais pas la même longueur,  lequel est avantagé par `IndexFlatIP` ? par la version normalisée ?

**7️⃣.** Dans un contexte de **recherche sémantique**, quelle mesure semble la plus pertinente : L2, IP brut ou IP + normalisation ?  


1️⃣ Non, les indices ne sont pas tous identiques :

IndexFlatL2 : id=11, id=5, id=7
IndexFlatIP (brut) : id=11, id=5, id=9
IndexFlatIP (normalisé) : id=11, id=5, id=7
Les deux premiers (id=11 et id=5) sont identiques pour les trois méthodes, mais le 3ème diffère : L2 et IP normalisé donnent id=7, tandis que IP brut donne id=9.

2️⃣ IndexFlatL2 mesure une distance euclidienne : plus la valeur est petite, plus les vecteurs sont proches. Les résultats sont donc triés par distance croissante (3.16 → 3.79 → 4.85).
IndexFlatIP mesure un produit scalaire (similarité) : plus la valeur est grande, plus les vecteurs sont similaires. Les résultats sont triés par score décroissant (2.19 → 2.04 → 1.76).

3️⃣ Distance euclidienne : mesure la longueur du segment entre deux points dans l'espace. $d(x,y) = \sqrt{\sum_i (x_i - y_i)^2}$. C'est une mesure de dissimilarité.
Produit scalaire : mesure l'alignement et les magnitudes des vecteurs. $x \cdot y = \sum_i x_i y_i = ||x|| \cdot ||y|| \cdot \cos(\theta)$. C'est une mesure de similarité.

4️⃣ Quand les vecteurs sont normalisés ($||x|| = ||y|| = 1$), le produit scalaire devient : $$x \cdot y = ||x|| \cdot ||y|| \cdot \cos(\theta) = 1 \cdot 1 \cdot \cos(\theta) = \cos(\theta)$$
Le produit scalaire mesure alors uniquement l'angle entre les vecteurs.

5️⃣ D'après les résultats :
Distance L2 : valeurs positives (3.16 à 4.85 dans cet exemple)
IndexFlatIP sans normalisation : valeurs variables (1.76 à 2.19 ici, peuvent être négatives)
IndexFlatIP avec normalisation : valeurs entre -1 et 1 (0.40 à 0.59 ici)

6️⃣ Sans normalisation (IP brut) : le vecteur le plus long est avantagé car le produit scalaire dépend des normes.
Avec normalisation : les deux obtiennent le même score car ils ont la même direction (même angle θ → même cos(θ)).

7️⃣ IP + normalisation (similarité cosinus) est la plus pertinente car elle mesure uniquement la proximité sémantique (direction des vecteurs) sans tenir compte de la longueur. En recherche sémantique, on veut comparer le "sens" des textes, pas leur magnitude.

In [ ]:
import numpy as np
import faiss

np.random.seed(42)

# --- Données synthétiques ---
n, d = 12, 8                 # 12 vecteurs, dimension 8
xb = np.random.randn(n, d).astype('float32')
xq = np.random.randn(1, d).astype('float32')

# Copie normalisée (pour simuler la similarité cosinus via IP)
xb_norm = xb.copy()
xq_norm = xq.copy()
faiss.normalize_L2(xb_norm)
faiss.normalize_L2(xq_norm)

# --- Index 1 : L2 ---
#Index calculant les distances euclidiennes
index_l2 = faiss.IndexFlatL2(d)
index_l2.add(xb)
D_l2, I_l2 = index_l2.search(xq, k=3)  # distances croissantes (plus petit = mieux)

# --- Index 2 : IP (produit scalaire brut, SANS normalisation) ---
index_ip_raw = faiss.IndexFlatIP(d)
index_ip_raw.add(xb)
S_ip_raw, I_ip_raw = index_ip_raw.search(xq, k=3)  # similarités décroissantes (plus grand = mieux)

# --- Index 3 : IP (COSINUS via normalisation L2) ---
index_ip_cos = faiss.IndexFlatIP(d)
index_ip_cos.add(xb_norm)
S_ip_cos, I_ip_cos = index_ip_cos.search(xq_norm, k=3)  # ≈ cos(x, y) ∈ [-1, 1]

def show(title, idxs, vals, is_distance=False):
    print(f"\n=== {title} ===")
    for rank, (i, v) in enumerate(zip(idxs[0], vals[0]), 1):
        metric = "distance" if is_distance else "score"
        print(f"{rank:>2}. id={i:>2}  {metric}={v:.6f}")

show("IndexFlatL2 (euclidien, plus petit = mieux)", I_l2, D_l2, is_distance=True)
show("IndexFlatIP (produit scalaire brut, plus grand = mieux)", I_ip_raw, S_ip_raw, is_distance=False)
show("IndexFlatIP (≈ cosinus avec normalisation, plus grand = mieux)", I_ip_cos, S_ip_cos, is_distance=False)


### 2- Générer un corpus artificiel pour tester les recherches

Le code suivant vise à générer un corpus artificiel mais significatif (environ 23000 phrases) qui nous permettra de tester les performances des index dans les questions suivantes.

In [ ]:
# ================================================================
# CORPUS FR SCALABLE (10k → 50k+) pour similarité sémantique
# Thèmes: CHAT↔SOFA, CHIEN↔JARDIN, METEO, CONFORT MEUBLE, BRUIT GÉNÉRIQUE
# Paramètre: TARGET = 20000, 50000, ...
# ================================================================
import random
from collections import OrderedDict

random.seed(1234)

TARGET = 20000   # <-- mets 10000, 20000, 50000 selon besoin
OVERSAMPLE = 1.6 # on suréchantillonne puis on déduplique (limite les collisions)
N_GEN = int(TARGET * OVERSAMPLE)

# Slots riches → grand espace combinatoire
LIEUX   = ["au salon","dans le salon","près de la fenêtre","dans la maison","au rez-de-chaussée",
           "dans la chambre","au grenier","au bureau","dans la cuisine","sur la terrasse"]
TEMPS   = ["ce matin","cet après-midi","ce soir","la nuit dernière","hier","demain matin",
           "en fin de journée","tôt le matin","vers midi","en début de soirée"]
INTENS  = ["un peu","plutôt","très","vraiment","assez","hautement","extrêmement","modérément"]
HEURES  = [f"{h:02d}h{m:02d}" for h in range(6, 23) for m in (0, 10, 20, 30, 40, 50)]
VILLES  = ["Paris","Lyon","Marseille","Lille","Bordeaux","Toulouse","Nice","Nantes","Rennes","Strasbourg",
           "Montpellier","Grenoble","Dijon","Angers","Clermont-Ferrand"]
JOURS   = ["lundi","mardi","mercredi","jeudi","vendredi","samedi","dimanche"]

# Vocab
cats = {
    "sujets": ["Un chat","Un félin","Le minou","Ce chat","Le félin","Un petit chat","Cette chatte"],
    "sieste": ["dort","se repose","somnole","fait la sieste","paresse","s’assoupit"],
    "supports": ["sur le canapé","sur le sofa","sur le divan","dans le fauteuil","sur le coussin du canapé","sur l’accoudoir"],
    "advs": ["calmement","tranquillement","profondément","doucement","au chaud","sans bruit"],
    "neg": ["ne dort pas","reste éveillé","saute partout","est agité","grimpe sur le dossier"],
}
dogs = {
    "sujets": ["Un chien","Le chien","Un jeune chien","Ce chien","Un border collie","Un berger","Un labrador","Un husky"],
    "verbes": ["court","joue","saute","gambade","file","galope","tourne"],
    "lieux": ["dans le jardin","au parc","sur la pelouse","près du portail","le long de l’allée","près du massif","autour des fleurs"],
    "advs": ["vite","joyeusement","sans s’arrêter","de bon matin","avec énergie","en rond"],
}
weather = {
    "ensoleille": [
        "Le soleil brille dans le ciel","La météo est ensoleillée","Ciel bleu et grand soleil",
        "Temps radieux et lumineux","L’après-midi s’annonce ensoleillée"
    ],
    "pluie": [
        "Il pleut légèrement","Pluie fine et continue","Le ciel est couvert avec des averses",
        "Temps humide avec de petites pluies","Des gouttes tombent sans arrêt"
    ],
    "vent": [
        "Le vent souffle fort","Rafales régulières sur la côte","Un vent frais traverse la ville",
        "Bourrasques en fin de journée","Un vent soutenu d’ouest"
    ],
}
sofa = {
    "meubles": ["canapé","sofa","divan","fauteuil","causeuse","banquette"],
    "adj": ["confortable","moelleux","douillet","plutôt ferme","neuf","profond","accueillant","large"],
    "intens": INTENS,
    "rooms": ["du salon","du séjour","de la maison","du coin TV","près de la bibliothèque","du bureau"],
    "phrases": [
        "Le {m} {r} est {i} {a}.",
        "Ce {m} {r} paraît {i} {a}.",
        "Je trouve le {m} {r} {i} {a}.",
        "Le {m} {r} semble {i} {a}.",
        "Le confort du {m} {r} est {i} {a}."
    ]
}
noise_pool = [
    "La base de données a été sauvegardée à {h}.",
    "Le train part à {h} depuis {v}.",
    "La soupe mijote sur le feu {t}.",
    "Le professeur corrige les copies {t}.",
    "Un vélo rouge est garé devant l’école {t}.",
    "Le code compile sans erreur {t}.",
    "La bibliothèque ferme à {h} le {j}.",
    "Le système a redémarré à {h}.",
    "Le colis est arrivé à {v} {t}.",
]

# Générateurs rapides (chaînes formatées) — pas de lourdes boucles imbriquées
def gen_cat(n):
    out = []
    for _ in range(n):
        subj = random.choice(cats["sujets"])
        verb = random.choice(cats["sieste"]) if random.random() > 0.12 else random.choice(cats["neg"])
        sent = f"{subj} {verb} {random.choice(cats['supports'])} {random.choice(cats['advs'])} " \
               f"{random.choice(LIEUX)} {random.choice(TEMPS)}."
        out.append(sent)
    return out

def gen_dog(n):
    out = []
    for _ in range(n):
        sent = f"{random.choice(dogs['sujets'])} {random.choice(dogs['verbes'])} " \
               f"{random.choice(dogs['lieux'])} {random.choice(dogs['advs'])} " \
               f"{random.choice(TEMPS)} {random.choice(LIEUX)}."
        out.append(sent)
    return out

def gen_weather(n):
    base = weather["ensoleille"] + weather["pluie"] + weather["vent"]
    out = []
    for _ in range(n):
        core = random.choice(base)
        sent = f"{core} à {random.choice(VILLES)} {random.choice(TEMPS)}."
        out.append(sent)
    return out

def gen_sofa(n):
    out = []
    for _ in range(n):
        phr = random.choice(sofa["phrases"])
        sent = phr.format(m=random.choice(sofa["meubles"]),
                          r=random.choice(sofa["rooms"]),
                          i=random.choice(sofa["intens"]),
                          a=random.choice(sofa["adj"]))
        out.append(sent)
    return out

def gen_noise(n):
    out = []
    for _ in range(n):
        tmpl = random.choice(noise_pool)
        sent = tmpl.format(h=random.choice(HEURES), v=random.choice(VILLES),
                           t=random.choice(TEMPS), j=random.choice(JOURS))
        out.append(sent)
    return out

# Répartition par thème (équilibrée et ajustable)
share = dict(cat=0.28, dog=0.22, met=0.22, sofa=0.16, noise=0.12)
n_cat   = int(N_GEN * share["cat"])
n_dog   = int(N_GEN * share["dog"])
n_met   = int(N_GEN * share["met"])
n_sofa  = int(N_GEN * share["sofa"])
n_noise = N_GEN - (n_cat + n_dog + n_met + n_sofa)

corpus = (
    gen_cat(n_cat) +
    gen_dog(n_dog) +
    gen_weather(n_met) +
    gen_sofa(n_sofa) +
    gen_noise(n_noise)
)

# Mélange + dédup (en conservant l’ordre mélangé)
random.shuffle(corpus)
corpus = list(OrderedDict.fromkeys(corpus))

# Si la dédup descend sous TARGET, on complète par un second tir en petite passe
while len(corpus) < TARGET:
    needed = TARGET - len(corpus)
    corpus += gen_noise(int(needed*1.2))  # thème "bruit" varie vite → peu de collisions
    random.shuffle(corpus)
    corpus = list(OrderedDict.fromkeys(corpus))

print("Taille finale du corpus :", len(corpus))

# Requêtes exemples
queries = [
    "Un chat fait la sieste sur le canapé.",
    "La météo est ensoleillée aujourd’hui.",
    "Un chien court dans le jardin.",
    "Le canapé est confortable."
]


**Réponses :**

**1️⃣** Mesures de latence ajoutées avec `time.perf_counter()`. Temps moyen par requête calculé en divisant le temps total par le nombre de requêtes.

**2️⃣** Comparaison sklearn vs IndexFlatIP (normalisé) : L'ordre des top-k est identique. Les scores sont identiques (à précision numérique près). IndexFlatIP est généralement plus rapide grâce aux optimisations C++ et SIMD. Même en force brute, FAISS est plus efficace que sklearn.

**3️⃣** Avec et sans normalisation : L'ordre des top-k peut être différent. Les scores sont très différents (produit scalaire brut vs cosinus). Sans normalisation, le produit scalaire favorise les vecteurs de grande magnitude.

**4️⃣** IndexFlatIP devient intéressant à partir de ~5000-10000 documents. L'avantage augmente avec la taille du corpus et la dimension des vecteurs.


### 3 - Comparer la recherche de similarité avec sklearn (cosine.similarity) et IndexFlatIP (avec et sans normalisation) 
1️⃣ Exécutez puis Modifiez le code suivant pour :     
    - Evaluer la latence (temps d'exacution) de chaque recherche selon les différents outils utilisés : cosine.similarity (sklearn), indexFlatIP (avec et sans normalisation)    
    <i>Indication : utilisez la méthode time.perf_counter() avant et aprés une recherche pour évaluer sa latence (temps de recherche)</i>  
    - Calculer le temps moyen de recherche par requete (temps_total/nb_queries)   
2️⃣ Observez les résultats obtenus avec sklearn (cosine.similarity) et IndexFlat IP (avec normalisation) :    
    - L'ordre des top_k est-il identique ?  
    - Les scores de similarité sont-ils identiques ?  
    - Les temps d'exécution sont-ils similaires ? si non quelle méthode semble plus performante?   
    - Sachant que IndexflatIP effectue une recherche en force brute de même que  cosine.similarity de sklearn, comment expliquer les variations de performances?    
3️⃣ Que remarquez-vous sur les scores obtenus avec IndexFlatIP avec et sans normalisation ?  
    - L'ordre des top_k est-il identique ?  
    - Les scores de similarité sont-ils identiques ?  
    - Comment expliquer ces résultats    
4️⃣ Faites varier la taille du corpus pour essayer d'identifier à partir de quelle taille l'indexflatIP devient intéressant par rapport à la recherche sklearn (cosine.similarity)  

<i><b>Remarque</b> : soit les vecteurs q=(1,1),v1​=(2,2),v2​=(1,0)    
Calculs des produits scalaires bruts (sans normalisation) :  q⋅v1​=1×2+1×2=4   q⋅v2​=1×1+1×0=1  
Calcul des similarités cosinus (avec normalisation ) :  ∣∣q∣∣=racine(2) ∣∣v1∣∣=2.racine(2) ∣∣v2∣∣=1    
    cos(q,v1)=  (q.v1) / (∣∣q∣∣ . ∣∣v1∣∣) = 1  
    cos(q,v2)=  (q.v2) / (∣∣q∣∣ . ∣∣v2∣∣) ≈ 0,707  


        

In [ ]:
# A COMPLETER
# --- Install (si besoin) ---
# !pip install -q sentence-transformers faiss-cpu scikit-learn

# import numpy as np
# from sentence_transformers import SentenceTransformer
# from sklearn.metrics.pairwise import cosine_similarity
# import faiss

# ----------------------
# Encodage SBERT
# ----------------------
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(model_name)

# Deux versions : normalisée (pour cosinus) et brute (pour tester l'effet de la norme)
emb_corpus_norm = model.encode(corpus, normalize_embeddings=True, batch_size=256, ).astype("float32")
emb_queries_norm = model.encode(queries, normalize_embeddings=True, batch_size=64, ).astype("float32")

emb_corpus_raw  = model.encode(corpus, normalize_embeddings=False).astype("float32")
emb_queries_raw = model.encode(queries, normalize_embeddings=False).astype("float32")

d = emb_corpus_norm.shape[1]

k = 4  # top-k

def print_topk(title, idxs, scores, labels):
    print(f"\n=== {title} ===")
    for qi, (I, S) in enumerate(zip(idxs, scores)):
        print(f"\nRequete: {queries[qi]}")
        for r, (i, s) in enumerate(zip(I[:k], S[:k]), 1):
            print(f"{r:>2}. {labels[i]}   (score={s:.4f})")

# -------------------------------------------------
# (a) sklearn cosine_similarity (embeddings normalisés)
# -------------------------------------------------
t0 = time.perf_counter()
cos_scores = cosine_similarity(emb_queries_norm, emb_corpus_norm)
I_sklearn = np.argsort(-cos_scores, axis=1)
latence_sklearn = time.perf_counter() - t0

# -------------------------------------------------
# (b) FAISS IndexFlatIP (sur vecteurs normalisés → IP == cosinus)
# -------------------------------------------------
index_ip = faiss.IndexFlatIP(d)
index_ip.add(emb_corpus_norm)
t0 = time.perf_counter()
S_ip, I_ip = index_ip.search(emb_queries_norm, k=len(corpus))
latence_IPnorm = time.perf_counter() - t0

# -------------------------------------------------
# (c) FAISS IndexFlatIP : sur vecteurs non normalisés
# -------------------------------------------------
index_ip2 = faiss.IndexFlatIP(d)
index_ip2.add(emb_corpus_raw)
t0 = time.perf_counter()
S_ip2, I_ip2 = index_ip2.search(emb_queries_raw, k=len(corpus))
latence_ip2 = time.perf_counter() - t0

print(f"[sklearn] latence = {latence_sklearn:.4f}s ({latence_sklearn/len(queries)*1000:.2f} ms/requête)")
print(f"[IndexFlatIP (norm)] latence = {latence_IPnorm:.4f}s ({latence_IPnorm/len(queries)*1000:.2f} ms/requête)")
print(f"[IndexFlatIP (sans norm)] latence = {latence_ip2:.4f}s ({latence_ip2/len(queries)*1000:.2f} ms/requête)")

print_topk("Sklearn cosine_similarity", I_sklearn, np.take_along_axis(cos_scores, I_sklearn, axis=1), corpus)
print_topk("FAISS IndexFlatIP (≈ cosinus car normalisé)", I_ip, S_ip, corpus)
print_topk("FAISS IndexFlatIP(sans normalisation)", I_ip2, S_ip2, corpus)


---
# Partie B — FAISS Index IVF (Inverted File Index)


**Réponses :**

**1️⃣** Le temps de recherche IVF est généralement plus court que IndexFlatIP, surtout pour de grands corpus. Pour de petits corpus (< 10k), la différence peut être négligeable.

**2️⃣** Les résultats (Top-k) ne sont pas toujours identiques. IndexFlatIP fait une recherche exhaustive tandis que IndexIVF fait une recherche approximative.

**3️⃣** IVF partitionne l'espace en `nlist` clusters et n'explore que `nprobe` clusters. Si un vecteur pertinent est dans un cluster non exploré, il ne sera pas trouvé. C'est le compromis vitesse ↔ précision.

**4️⃣** Augmenter `nlist` crée un partitionnement plus fin avec une meilleure précision potentielle, mais augmente le temps de recherche si `nprobe` augmente aussi. Il faut équilibrer `nlist` et `nprobe` selon la taille du corpus.


### 1 - Comparer IndexFlatIP et Index IVF 

Complétez le code suivant :
  
- complétez les instructions de création d'un index IVF
- comparez les résultats obtenus avec ceux d'un indexFlatIP (avec normalisation)

Interprétez les résultats obtenus en répondant aux questions suivantes :  
    1️⃣ Le temps de recherche IVF est-il plus court que celui du IndexFlatIP ?  
    2️⃣ Les résultats (Top-k) sont-ils identiques ?  
    3️⃣ Pourquoi peut-on avoir des différences de scores ou d’ordre ?  
    4️⃣ Que se passerait-il si l’on augmentait nlist (le nombre de clusters) ? Testez  



**Réponses :**

**1️⃣** La fonction `recall_at_k` calcule la proportion de résultats pertinents retrouvés par l'index approximatif (IVF) parmi les k premiers résultats de l'index exact (Flat). Formule : $\text{recall@k} = \frac{|\text{Top-k}_{\text{IVF}} \cap \text{Top-k}_{\text{Flat}}|}{k}$

**2️⃣** La liste de recall indique la qualité de l'approximation pour chaque requête. Un recall@5 de 1.0 signifie que tous les top-5 de l'index exact ont été retrouvés (100%). Un recall@5 de 0.8 signifie que 4 des 5 meilleurs résultats ont été retrouvés (80%).

**3️⃣** Influence de `nprobe` : nprobe = 1 est très rapide mais recall faible. Augmenter nprobe (2, 3, 5, 10) améliore le recall mais augmente la latence. Trade-off entre précision et vitesse.

Influence de `nlist` : nlist petit (ex: 20) donne des clusters larges et moins précis. nlist optimal (ex: 100-200) offre un bon équilibre. nlist trop grand (ex: 1000) augmente les temps d'entraînement et de recherche.

**4️⃣** Configuration optimale pour corpus ~20k : `nlist = 100-150` et `nprobe = 5-10`. Règle empirique : $\text{nlist} \approx \sqrt{N}$ où N = taille du corpus. Pour recall@5 > 0.95, nprobe ≈ 10-15% de nlist.


In [ ]:
# A COMPLETER 

# import time

d = emb_corpus_norm.shape[1]
nlist = 100
quantizer = faiss.IndexFlatIP(d)

index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)
index_ivf.train(emb_corpus_norm)
index_ivf.add(emb_corpus_norm)

index_flat = faiss.IndexFlatIP(d)
index_flat.add(emb_corpus_norm)

t0 = time.perf_counter()
D_flat, I_flat = index_flat.search(emb_queries_norm, k=5)
t1 = time.perf_counter()
print(f"⏱️  Temps de recherche (IndexFlatIP) : {t1 - t0:.4f} s\n")

index_ivf.nprobe = 1
t0 = time.perf_counter()
D_ivf, I_ivf = index_ivf.search(emb_queries_norm, k=5)
t1 = time.perf_counter()
print(f"⏱️ Temps de recherche IVF : {t1 - t0:.4f} s")

print(f"\nRequête: {queries[0]}")
print("\n--- Résultats IndexIVFFlat ---")
for rank, (idx, score) in enumerate(zip(I_ivf[0], D_ivf[0]), 1):
    print(f"{rank}. {corpus[idx]}  (cos={score:.3f})")
print("\n--- Résultats IndexFlatIP ---")
for rank, (idx, score) in enumerate(zip(I_flat[0], D_flat[0]), 1):
    print(f"{rank}. {corpus[idx]}  (cos={score:.3f})")


### 2 - Evaluer la pertinence d'une recherche  IVF en calculant le recall@

Testez le code suivant et répondez au questions suivantes :  
    1️⃣ Que calcule la fonction recall_at_k ?  
    2️⃣ Interprétez les résultats affichés :  
        - Quelles indications nous donne la liste de recall ?  
    3️⃣ nlist représente la liste des clusters et nprobe le nombre de clusters explorés lors de l'exécution d'une requete  
        - Tentez d'augmenter nprobe : 2 puis 3 ect...  
        - Interprétez les résultats obtenus en expliquant l'influence de ce paramètre sur le recall@k et la latence  
        - Tentez à présent de modifier nlist  
        - Interpretez les résultats obtenus en expliquant l'influence de ce paramètre sur le recall@k et la latence  
    4️⃣ Proposez une version optimale de valeurs pour nprobe et nlist  



In [ ]:
def recall_at_k(true_idx, approx_idx, k=10):
    # true_idx : indices triés décroissant (meilleurs d'abord) renvoyé par l'index de référence
    # approx_idx : indices renvoyés par l'index approximatif
    return len(set(true_idx[:k]).intersection(set(approx_idx[:k]))) / k

# Données
d = emb_corpus_norm.shape[1] #on reutilise le corpus normalisé de la question précédente

# Recherche avec Index Flat (référence pour évaluer la recherche IVF) 
index_flat = faiss.IndexFlatIP(d)
index_flat.add(emb_corpus_norm)
t0 = time.perf_counter()
D_flat, I_flat = index_flat.search(emb_queries_norm, k=5)
latence_flat = time.perf_counter()-t0

# 🧩 création d'un index IVF 
nlist=100
index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)
index_ivf.train(emb_corpus_norm)   # sur le corpus
index_ivf.add(emb_corpus_norm)     # ajoute les vecteurs

# Recherche avec IndexFlatIVF
index_ivf.nprobe = 1 # valeur par défaut
t0 = time.perf_counter()
D_ivf, I_ivf = index_ivf.search(emb_queries_norm, k=5)
latence_ivf = time.perf_counter()-t0

list_recall=[]
#Calcul des recall@k
for i  in range(len(emb_queries_norm)):
    #pour chaque requete
    list_recall.append(recall_at_k(I_flat[i],I_ivf[i],k=5 ))

print("Liste des recall =", list_recall)
moy_recall=sum(list_recall)/len(list_recall)

print("recall@k moyen :", moy_recall)
print("latence ivf :", latence_ivf)
print("latence flat :", latence_flat)


---
# Partie COURS  — exemples code **CH4**


### Exemple de requete de proximité géographique avec IndexFlatL2

In [ ]:
import numpy as np, faiss

# Positions 3D de capteurs (x, y, z)
positions = np.array([
    [0.0, 0.0, 0.0],   # capteur A
    [1.0, 0.0, 0.0],   # capteur B
    [0.0, 1.0, 0.0],   # capteur C
    [2.0, 1.0, 0.0],   # capteur D
], dtype='float32')

# Requête : un point d’intérêt
q = np.array([[1.2, 0.7, 0.0]], dtype='float32')

# Index euclidien
index = faiss.IndexFlatL2(3)
index.add(positions)
D, I = index.search(q, k=2)

print("Indices des capteurs les plus proches:", I)
print("Distances par rapport à ces capteurs :",D)





### Exemple de Requête de Recherche de similarité avec IndexFlatIP

In [ ]:
# --- Corpus de 3 "couleurs" en dimension 3 (R, G, B) ---
E = np.array([
    [1.0, 0.0, 0.0],   # Rouge pur
    [0.0, 1.0, 0.0],   # Vert pur
    [0.0, 0.0, 1.0],   # Bleu pur
], dtype='float32')
# --- Création d'un index Flat IP (produit scalaire) dim=3---
index = faiss.IndexFlatIP(3)
index.add(E)
# --- Requête : une couleur orangée = mélange de rouge et un peu de vert 
q = np.array([[0.8, 0.2, 0.0]], dtype='float32')
# --- Recherche top-2 voisins ---
D, I = index.search(q, k=2)

print("Voisins (sans normalisation)")
print("Indices: ", I, "Scores (produit scalaire):", D)

#avec vecteurs normalisés
E_norm = E.copy()
q_norm = q.copy()
faiss.normalize_L2(E_norm) 
# normalise chaque ligne à ||x||=1
faiss.normalize_L2(q_norm)
index_norm = faiss.IndexFlatIP(3)
index_norm.add(E_norm)
# --- Recherche top-2 voisins ---
D_norm, I_norm = index_norm.search(q_norm, k=2)

print("Voisins(avec normalisation)")
print("Indices: ", I_norm, "Scores (similarité cosinus):", D_norm)
